# CLASS 5
# Machine Learning Package - Backward Propagation

In [1]:
import numpy as np

## Exercise — One training step by hand

Network from last class (2 → 3 → 1, no activations), with $h = [0.8, -0.2, 0.6]$.

2nd layer: $W_2 = [0.2, -0.3, 0.1]^T$, $b_2 = 0$. Target $y = 1$, $\alpha = 0.1$.

Loss: $E = (y - \hat{y})^2$, so $\frac{dE}{d\hat{y}} = 2(\hat{y} - y)$.

In [2]:
x  = np.array([[1, 2]])
W1 = np.array([[0.1,  0.2, -0.1],
               [0.3, -0.2,  0.4]])
b1 = np.array([[0.1, 0, -0.1]])

W2 = np.array([[0.2], [-0.3], [0.1]])
b2 = np.zeros((1, 1))

y     = 1
alpha = 0.1

# 1. Forward: y_hat and loss
h     = np.dot(x, W1) + b1
y_hat = np.dot(h, W2) + b2
E     = (y - y_hat) ** 2

print('h:    ', h)
print('y_hat:', y_hat)
print('Loss: ', E)

# 2. dE/dy_hat
dE_dyhat = 2 * (y_hat - y)
print('dE/dy_hat:', dE_dyhat)

# 3. Gradients of the 2nd layer and error passed to the 1st layer
dE_dW2 = np.dot(h.T, dE_dyhat)    
dE_db2 = dE_dyhat           
dE_dh  = np.dot(dE_dyhat, W2.T)      

print('dE/dW2:', dE_dW2.T)
print('dE/db2:', dE_db2)
print('dE/dh: ', dE_dh)

# 4. Update W2 and b2
W2 = W2 - alpha * dE_dW2
b2 = b2 - alpha * dE_db2

print('W2:', W2.T)
print('b2:', b2)

# 5. Update W1 and b1, forward again and compare the loss
dE_dW1 = np.dot(x.T, dE_dh)        
dE_db1 = dE_dh                       

W1 = W1 - alpha * dE_dW1
b1 = b1 - alpha * dE_db1

h_new     = np.dot(x, W1) + b1
y_hat_new = np.dot(h_new, W2) + b2
E_new     = (y - y_hat_new) ** 2

print('W1:', W1)
print('b1:', b1)
print()
print('y_hat: ', y_hat[0, 0], '->', y_hat_new[0, 0])
print('Loss:  ', E[0, 0], '->', E_new[0, 0])

h:     [[ 0.8 -0.2  0.6]]
y_hat: [[0.28]]
Loss:  [[0.5184]]
dE/dy_hat: [[-1.44]]
dE/dW2: [[-1.152  0.288 -0.864]]
dE/db2: [[-1.44]]
dE/dh:  [[-0.288  0.432 -0.144]]
W2: [[ 0.3152 -0.3288  0.1864]]
b2: [[0.144]]
W1: [[ 0.1288  0.1568 -0.0856]
 [ 0.3576 -0.2864  0.4288]]
b1: [[ 0.1288 -0.0432 -0.0856]]

y_hat:  0.28 -> 0.7295564800000001
Loss:   0.5184 -> 0.07313969750999037


## Exercise — DenseLayer.backward_propagation
**Gradient check: compare the analytic gradient dE/dW with the numerical one (finite differences).**
SGD and MSE only come next class, so we use a "fake" optimizer that does not change the weights and an MSE in numpy.

In [3]:
from si.neural_networks.layers import DenseLayer


class NoUpdate:                                    
    def update(self, w, grad):
        return w


mse            = lambda Y, P: np.mean((Y - P) ** 2)
mse_derivative = lambda Y, P: 2 * (P - Y) / Y.size

np.random.seed(0)

X, Y = np.random.rand(4, 3), np.random.rand(4, 2)

layer    = DenseLayer(2, (3,)).initialize(NoUpdate())
out      = layer.forward_propagation(X, training = True)
analytic = X.T @ mse_derivative(Y, out)

numeric, eps = np.zeros_like(layer.weights), 1e-6

for i in range(3):
    for j in range(2):
        layer.weights[i, j] += eps
        lp = mse(Y, layer.forward_propagation(X, True))
        layer.weights[i, j] -= 2 * eps
        lm = mse(Y, layer.forward_propagation(X, True))
        layer.weights[i, j] += eps
        numeric[i, j] = (lp - lm) / (2 * eps)

print(np.abs(numeric - analytic).max())

1.513400516017782e-11


## Exercise — Activation layers (ReLU and Sigmoid)
**Check the derivatives against the numerical derivative (finite differences).**

In [4]:
from si.neural_networks.activation import ReLUActivation, SigmoidActivation

x, eps = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7]), 1e-6

for act in [SigmoidActivation(), ReLUActivation()]:
    num = (act.activation_function(x + eps)
           - act.activation_function(x - eps)) / (2 * eps)
    print(act.layer_name(), np.abs(num - act.derivative(x)).max())   

SigmoidActivation 6.56039597646263e-11
ReLUActivation 1.397779669787269e-10


## Exercise 12 — Dropout layer
### 12.2) Test the layer with a random input
**Small example: which neurons are dropped and how the kept ones are scaled.**

In [5]:
from si.neural_networks.layers import Dropout

np.random.seed(1)

X_small = np.random.rand(2, 5)
dropout = Dropout(probability = 0.5)

print('Input:')
print(X_small.round(2))
print('Mask:')
out_small = dropout.forward_propagation(X_small, training = True)
print(dropout.mask)
print('Output (training):')
print(out_small.round(2))

Input:
[[0.42 0.72 0.   0.3  0.15]
 [0.09 0.19 0.35 0.4  0.54]]
Mask:
[[0 1 0 1 0]
 [1 0 1 0 0]]
Output (training):
[[0.   1.44 0.   0.6  0.  ]
 [0.18 0.   0.69 0.   0.  ]]


**Check the expected behaviour on a larger random input.**

In [6]:
np.random.seed(0)

X       = np.random.rand(1000, 10)
dropout = Dropout(probability = 0.5)

out  = dropout.forward_propagation(X, training = True)
kept = out != 0

print('Fraction of dropped neurons:', np.mean(out == 0))                         
print('Kept values scaled by 1/(1-p):', np.allclose(out[kept], X[kept] * 2))     
print('Mean input: ', X.mean().round(4))
print('Mean output:', out.mean().round(4))                                      

out_inference = dropout.forward_propagation(X, training = False)
print('Inference returns the input unchanged:', np.array_equal(out_inference, X)) 

input_error = dropout.backward_propagation(np.ones_like(X))
print('Backward zeros where the mask is 0:', np.array_equal(input_error == 0, dropout.mask == 0))  

dropout.set_input_shape((10,))
print('Output shape:', dropout.output_shape())   
print('Parameters:  ', dropout.parameters())    

Fraction of dropped neurons: 0.5065
Kept values scaled by 1/(1-p): True
Mean input:  0.4965
Mean output: 0.4925
Inference returns the input unchanged: True
Backward zeros where the mask is 0: True
Output shape: (10,)
Parameters:   0


## Exercise 13 — TanhActivation and SoftmaxActivation
### 13.1) TanhActivation

In [ ]:
from si.neural_networks.activation import TanhActivation, SoftmaxActivation

x, eps = np.array([-2.5, -1.2, -0.3, 0.4, 1.1, 2.7]), 1e-6

tanh = TanhActivation()

num = (tanh.activation_function(x + eps)
       - tanh.activation_function(x - eps)) / (2 * eps)

print('Derivative check:', np.abs(num - tanh.derivative(x)).max())                 
print('Range [-1, 1]:   ', tanh.activation_function(np.array([-100., 0., 100.]))) 

Derivative check: 3.338995746560158e-11
Range [-1, 1]:    [-1.  0.  1.]


### 13.2) SoftmaxActivation

In [8]:
softmax = SoftmaxActivation()

X = np.array([[1.,  2.,  3.],
              [1.,  1.,  1.],
              [0., -5., 10.]])

P = softmax.activation_function(X)

print('Softmax:')
print(P.round(4))
print('Row sums:', P.sum(axis = 1))                                              

big = np.array([[1000., 1001., 1002.]])
print('Large values:', softmax.activation_function(big))               

Softmax:
[[0.09   0.2447 0.6652]
 [0.3333 0.3333 0.3333]
 [0.     0.     1.    ]]
Row sums: [1. 1. 1.]
Large values: [[0.09003057 0.24472847 0.66524096]]
